In [1]:
import mlflow
import os
import pandas as pd
from dotenv import load_dotenv

from mlflow.genai import scorer
from datasets import load_dataset
from mlflow.genai.datasets import create_dataset



load_dotenv()

os.environ["MLFLOW_TRACKING_URI"] = "http://localhost:5000"
os.environ["MLFLOW_EXPERIMENT_NAME"] = "log_model_with_prompt"
# mlflow.set_experiment("log_model_with_prompt")

d:\youtube\TheAIGuy\NLP\mlflow_examples\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset = load_dataset("ag_news", split="train")
df = dataset.to_pandas()

df["label"] = df["label"].map({0: "World", 1: "Sports", 2: "Business", 3: "Science"})

df = df.sample(frac=1).reset_index(drop=True)
df.head()

,text,label
0,NHL: Pucks iced The National Hockey League shu...,Sports
1,"SpaceShipOne successful again MOJAVE, Calif. -...",Science
2,AP: Kids Left in Africa Begged for Change (AP)...,World
3,Passport Privacy Protection? Nope The Bush adm...,Science
4,"Earnings, Oil Prices Drive Stocks Lower Invest...",Business


In [3]:
# Create a detailed prompt for classification
prompt = """
You are a helpful assistant that can classify news articles into one of the following categories:
- World
- Sports
- Business
- Science
Article: {article}
"""

initial_prompt = mlflow.genai.register_prompt(
    name="news_classifier",
    template=prompt,
)

2025/11/10 17:00:06 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: news_classifier, version 1


In [4]:
prompt_uri = "prompts:/news_classifier/1"

NUM_SAMPLES = 3
train_data = []
for i in range(NUM_SAMPLES):
    article = df.iloc[i]["text"]
    expected = df.iloc[i]["label"]
    eval_dict = {
        "inputs": {"article": article, "prompt_uri": prompt_uri},
        "expectations": {"expected_response": expected},
    }
    train_data.append(eval_dict)

train_data[0]

{'inputs': {'article': 'NHL: Pucks iced The National Hockey League shut down at midnight Wednesday night, locking out players from training camps and signaling that owners are willing to cancel the ',
  'prompt_uri': 'prompts:/news_classifier/1'},
 'expectations': {'expected_response': 'Sports'}}

In [5]:
df = pd.DataFrame(train_data)
df

,inputs,expectations
0,{'article': 'NHL: Pucks iced The National Hock...,{'expected_response': 'Sports'}
1,{'article': 'SpaceShipOne successful again MOJ...,{'expected_response': 'Science'}
2,{'article': 'AP: Kids Left in Africa Begged fo...,{'expected_response': 'World'}


In [6]:
with mlflow.start_run(run_name="langchain_model"):
    model_info = mlflow.pyfunc.log_model(
        name="news_classifier_with_prompt",
        python_model="lc_model_with_prompt.py",
        prompts=[prompt_uri],
        input_example=[train_data[0]["inputs"]],
    )

2025/11/10 17:00:31 INFO mlflow.models.signature: Running the predict function to generate output based on input example


Loading context
Received Input:  {'article': 'NHL: Pucks iced The National Hockey League shut down at midnight Wednesday night, locking out players from training camps and signaling that owners are willing to cancel the ', 'prompt_uri': 'prompts:/news_classifier/1'}


2025/11/10 17:01:17 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


Loading context
Received Input:  {'article': 'NHL: Pucks iced The National Hockey League shut down at midnight Wednesday night, locking out players from training camps and signaling that owners are willing to cancel the ', 'prompt_uri': 'prompts:/news_classifier/1'}


2025/11/10 17:01:30 INFO mlflow.models.model: Found the following environment variables used during model inference: [GOOGLE_API_KEY, OPENAI_API_KEY]. Please check if you need to set them when deploying the model. To disable this message, set environment variable `MLFLOW_RECORD_ENV_VARS_IN_MODEL_LOGGING` to `false`.


🏃 View run langchain_model at: http://localhost:5000/#/experiments/2/runs/6d95cb1cde26408b995b8a936237cf6c
🧪 View experiment at: http://localhost:5000/#/experiments/2


In [7]:
model = mlflow.pyfunc.load_model(model_uri=model_info.model_uri)

Loading context


In [8]:
dataset = create_dataset(
    name="ag_news_classifier_sample",
    experiment_id=mlflow.get_experiment_by_name("log_model_with_prompt").experiment_id,
    tags={"stage": "testing"},
)

In [9]:
model.predict([train_data[0]["inputs"]])

Received Input:  {'article': 'NHL: Pucks iced The National Hockey League shut down at midnight Wednesday night, locking out players from training camps and signaling that owners are willing to cancel the ', 'prompt_uri': 'prompts:/news_classifier/1'}


['**Sports**']

In [ ]:
# model.predict(["str"])

In [10]:
def predict_fn(article, prompt_uri):
    print("Article: ", article)
    response = model.predict([{"article": article, "prompt_uri": prompt_uri}])
    return response

In [11]:
predict_fn(train_data[0]["inputs"]["article"], train_data[0]["inputs"]["prompt_uri"])

Article:  NHL: Pucks iced The National Hockey League shut down at midnight Wednesday night, locking out players from training camps and signaling that owners are willing to cancel the 
Received Input:  {'article': 'NHL: Pucks iced The National Hockey League shut down at midnight Wednesday night, locking out players from training camps and signaling that owners are willing to cancel the ', 'prompt_uri': 'prompts:/news_classifier/1'}


['**Sports**']

In [16]:
dataset.merge_records(train_data)

In [ ]:
@scorer
def exact_match(outputs, expectations):
    expectations = expectations["expected_response"]
    return outputs[0] == expectations


with mlflow.start_run(run_name="evaluation"):
    results = mlflow.genai.evaluate(
        data=dataset,
        scorers=[exact_match],
        predict_fn=predict_fn,
        model_id=model.model_id,
    )

2025/11/10 17:05:57 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.


Article:  SpaceShipOne successful again MOJAVE, Calif. -- By the time test pilot Brian Binnie shut down the rocket engine on SpaceShipOne Monday morning, the view out his cockpit windows had turned from bright, desert blue to black.
Received Input:  {'article': 'SpaceShipOne successful again MOJAVE, Calif. -- By the time test pilot Brian Binnie shut down the rocket engine on SpaceShipOne Monday morning, the view out his cockpit windows had turned from bright, desert blue to black.', 'prompt_uri': 'prompts:/news_classifier/1'}


2025/11/10 17:05:57 INFO mlflow.tracking.fluent: Active model is set to the logged model with ID: m-2fbec1524c8f46f5a2496c7bb398e797
2025/11/10 17:05:57 INFO mlflow.tracking.fluent: Use `mlflow.set_active_model` to set the active model to a different one if needed.


Article: Article:  AP: Kids Left in Africa Begged for Change (AP) AP - Allegedly abandoned by their American mother in Africa, seven children from Texas begged small change to buy food and shuttled from a neglectful stranger's care to a concrete-block orphanage, Nigerians said Thursday.
Received Input:  {'article': "AP: Kids Left in Africa Begged for Change (AP) AP - Allegedly abandoned by their American mother in Africa, seven children from Texas begged small change to buy food and shuttled from a neglectful stranger's care to a concrete-block orphanage, Nigerians said Thursday.", 'prompt_uri': 'prompts:/news_classifier/1'}
 SpaceShipOne successful again MOJAVE, Calif. -- By the time test pilot Brian Binnie shut down the rocket engine on SpaceShipOne Monday morning, the view out his cockpit windows had turned from bright, desert blue to black.
Received Input:  {'article': 'SpaceShipOne successful again MOJAVE, Calif. -- By the time test pilot Brian Binnie shut down the rocket engine o

Evaluating: 100%|██████████| 3/3 [Elapsed: 00:06, Remaining: 00:00] 


In [20]:
annotated_traces  = mlflow.search_traces(model_id=model.model_id, max_results=100, return_type="list")

In [21]:
dataset.merge_records(annotated_traces)